# Tutorial: Survey Weights in socialmixr

This notebook is a reference guide to the **participant weights** offered by the R package [socialmixr](https://github.com/epiforecasts/socialmixr). For each weight it covers:

1. what the weight corrects for,
2. how it is defined mathematically, and
3. how it is applied in R.

The notebook first works through one concrete example, day-of-week weights, and then presents the general weighting setup and the remaining weight types.

> **Scope.**
> - Everything here describes socialmixr **0.6.0.9000** (development version, commit `b0d7134`).
> - The R code is shown in markdown blocks **for reference only**; nothing in this notebook runs R.
> - The numbers in the worked examples were computed in R on the POLYMOD United Kingdom data with `age_limits = c(0, 5, 15)`.

## Contents

0. [Notation](#0.-Notation)
1. [Why weight contact surveys?](#1.-Why-weight-contact-surveys?)
2. [Introductory example with day-of-week weights](#2.-Introductory-example-with-day-of-week-weights)
3. [The general weighting setup](#3.-The-general-weighting-setup)
4. [The weight types](#4.-The-weight-types)
5. [Caveats and limitations](#5.-Caveats-and-limitations)
6. [References](#6.-References)

## 0. Notation

| Symbol                              | Meaning                                                                                                |
|-------------------------------------|--------------------------------------------------------------------------------------------------------|
| $i, j$                              | participants (survey respondents); $j$ is used as a second participant index inside sums               |
| $a, b$                              | single-year ages: $a$ for a participant, $b$ for a contact                                             |
| $c, d$                              | age groups: $c$ for a participant (a row of the contact matrix), $d$ for a contact (a column)          |
| $a_i$                               | single-year age of participant $i$                                                                     |
| $\mathcal{N}_c$                     | set of participants in age group $c$                                                                   |
| $n_c = \lvert \mathcal{N}_c \rvert$ | number of participants in age group $c$                                                                |
| $n_a$                               | number of participants of single-year age $a$                                                          |
| $N$                                 | total number of participants                                                                           |
| $y_{ib}$                            | number of contacts of single-year age $b$ reported by participant $i$                                  |
| $y_{id} = \sum_{b \in d} y_{ib}$    | number of contacts in age group $d$ reported by participant $i$                                        |
| $m_{cd}$                            | estimated mean number of contacts that a participant in age group $c$ has with people in age group $d$ |
| $w_i$                               | weight of participant $i$                                                                              |
| $f^{(k)}_i$                         | the factor that the $k$-th weighting step multiplies into $w_i$                                        |
| $P_a$, $P_c$, $P$                   | population size at single-year age $a$, in age group $c$, and in total                                 |

All weights in socialmixr are **participant-level**. Each contact inherits the weight of the participant who reported it. There are no contact-level weights.

## 1. Why weight contact surveys?

Contact surveys are diary studies where each participant records the people they met on a assigned day. Two common sources of bias when estimating social contact estimates (not limited to contact matrices) are:

- **Day of week.** Contact behaviour differs sharply between weekdays and weekends. If the survey days are not spread in the ratio 5 weekdays : 2 weekend days, an unweighted average does not represent a typical week.
- **Age composition.** The age profile of the sample differs from that of the population. Contact matrices are already stratified by age group $c$, so the imbalance that matters is the one **within** each age group. For example, a band $[15, \infty)$ in which 20-year-olds are over-represented relative to 70-year-olds.

socialmixr corrects for these with **post-stratification-style multiplicative weights**. It also offers a general mechanism, `weigh()`, for user-defined weights.

**The assumption behind weighting.** Weighting by an extra variable (day of week, single-year age within a band, country, …) assumes that contact behaviour **differs across its categories** within age group $c$. The unweighted and the weighted $m_{cd}$ both average the category-specific mean contacts with age group $d$; they differ only in the shares given to each category (sample shares versus target shares). If participants in every category had the same mean contacts, for example if weekday and weekend contacts were the same, the two averages would coincide and no weighting would be needed. Weighting would then only add variance. Weighting also has no effect when the sample shares already match the target shares.

## 2. Introductory example with day-of-week weights

Before presenting a general formula for constructing weights, we begin with a simple example on day-of-week weights. In POLYMOD UK study, 1011 participants took part:

- 709 reported on a weekday,
- 258 reported on a weekend day,
- 44 have no recorded survey day.

Among the 967 participants with a known day, the weekday share is $709 / 967 = 0.733$, above the weekly share $5/7 \approx 0.714$. An unweighted average therefore leans slightly towards weekday behaviour. Its target is the **weekly average**, which counts weekdays and weekend days in the ratio 5 : 2.

Let $D_i \in \{0, 1, \dots, 6\}$ be participant $i$'s survey day, with 0 = Sunday, …, 6 = Saturday (the POLYMOD convention). Split the participants into three groups:

- The set of participant who reported on a weekday: $\mathcal{W} = \{ i : D_i \in \{1,\dots,5\} \}$, with $n_{\text{wd}} = \lvert \mathcal{W} \rvert$;
- weekend: $\mathcal{E} = \{ i : D_i \in \{0, 6\} \}$, with $n_{\text{we}} = \lvert \mathcal{E} \rvert$;
- missing day: $\mathcal{M} = \{ i : D_i \text{ is NA} \}$, so that $N = n_{\text{wd}} + n_{\text{we}} + \lvert \mathcal{M} \rvert$.

Each participant gets the factor

$$
f^{\text{dow}}_i =
\begin{cases}
5 / n_{\text{wd}}, & i \in \mathcal{W}, \\[2pt]
2 / n_{\text{we}}, & i \in \mathcal{E}, \\[2pt]
7 / N, & i \in \mathcal{M}.
\end{cases}
$$

The weekday participants then carry a total weight of $\sum_{i \in \mathcal{W}} f^{\text{dow}}_i = 5$, and the weekend participants a total of $\sum_{i \in \mathcal{E}} f^{\text{dow}}_i = 2$. Participants with a missing day get the average weight $7/N$.

For POLYMOD UK this gives:

| Participants               | Count                 | Factor $f^{\text{dow}}_i$ |
|----------------------------|-----------------------|---------------------------|
| weekday, $\mathcal{W}$     | $n_{\text{wd}} = 709$ | $5/709 = 0.00705$         |
| weekend, $\mathcal{E}$     | $n_{\text{we}} = 258$ | $2/258 = 0.00775$         |
| missing day, $\mathcal{M}$ | $44$                  | $7/1011 = 0.00692$        |

Weekend respondents are up-weighted relative to weekday respondents by the factor $0.00775 / 0.00705 = 1.10$.

`socialmixr` estimates each entry of the contact matrix as a **weighted mean over the participants in age group $c$**:

<a id="wd-weighted-intensity"></a>
$$
m_{cd} \;=\; \frac{\sum_{i \in \mathcal{N}_c} w_i \, y_{id}}{\sum_{i \in \mathcal{N}_c} w_i},
\qquad\text{here with } w_i = f^{\text{dow}}_i. \tag{1}
$$

**Interpretation.** To see what this does, split $\mathcal{N}_c$ by survey day. For clarity of exposition, we will also forget about the respondents with missing day-of-week. Write $n_{c,\text{wd}} = \lvert \mathcal{N}_c \cap \mathcal{W} \rvert$ and $n_{c,\text{we}} = \lvert \mathcal{N}_c \cap \mathcal{E} \rvert$ (the number of respondents in age group $c$ that reported contacts on a weekday and weekends, respectively). Also, define the *unweighted* means

$$
\bar y^{\text{wd}}_{cd} = \frac{1}{n_{c,\text{wd}}} \sum_{i \in \mathcal{N}_c \cap \mathcal{W}} y_{id},
\qquad
\bar y^{\text{we}}_{cd} = \frac{1}{n_{c,\text{we}}} \sum_{i \in \mathcal{N}_c \cap \mathcal{E}} y_{id}.
$$

All participants in a cell ($(c, \text{wd})$ or $(c, \text{we})$) share the same factor, so substituting into [(1)](#wd-weighted-intensity) gives
$$
m_{cd} \;=\; \lambda^{\text{wd}}_c \, \bar y^{\text{wd}}_{cd} \;+\; \lambda^{\text{we}}_c \, \bar y^{\text{we}}_{cd},
$$

with mixing weights that sum to one:

$$
\lambda^{\text{wd}}_c = \frac{n_{c,\text{wd}} \cdot 5 / n_{\text{wd}}}{Z_c}, \qquad
\lambda^{\text{we}}_c = \frac{n_{c,\text{we}} \cdot 2 / n_{\text{we}}}{Z_c}, \qquad
$$

$$
Z_c = \frac{n_{c,\text{wd}}}{n_{\text{wd}}}5 \;+\; \frac{n_{c,\text{we}}}{n_{\text{we}}} 2.
$$

Without weights, the mixing weights would simply be the sample shares $n_{c,\text{wd}} / n_c$, $n_{c,\text{we}} / n_c$ and $n_{c,\text{na}} / n_c$. **Weighting replaces the sample's day-of-week mix within age group $c$ with the mix $\lambda_c$.**

### In R

```r
library(socialmixr)
data(polymod)

uk <- polymod[country == "United Kingdom"] |>
  assign_age_groups(age_limits = c(0, 5, 15))

uk_w <- uk |> weigh_by_dayofweek()        # adds / multiplies the `weight` column
uk_w$participants[, .N, by = .(dayofweek, weight)]   # inspect the factors

uk_w |> compute_matrix()                  # weighted contact matrix m_cd
```
Equivalently:
```r
# General form, see Section 4; use this if your data code days differently
uk |> weigh("dayofweek", target = c(5, 2), groups = list(1:5, c(0, 6)))
```

**Notes.**
- If the participant table has no `dayofweek` column, `weigh_by_dayofweek()` warns and returns the survey unchanged.
- If your data codes days as 1 = Monday … 7 = Sunday, use the general `weigh()` form with matching `groups`. Otherwise Sunday (7) matches no group and silently gets the average weight $7/N$.

### Caveat: The weights are set across the whole sample, not within each age group
Among participants with a known day, the weighted weekday share in age group $c$ is
$$
s_c \;=\; \frac{\lambda^{\text{wd}}_c}{\lambda^{\text{wd}}_c + \lambda^{\text{we}}_c}
\;=\; \frac{n_{c,\text{wd}} \cdot 5 / n_{\text{wd}}}{n_{c,\text{wd}} \cdot 5 / n_{\text{wd}} \;+\; n_{c,\text{we}} \cdot 2 / n_{\text{we}}} .
$$
It equals $5/7$ exactly when $n_{c,\text{wd}} / n_{c,\text{we}} = n_{\text{wd}} / n_{\text{we}}$, i.e. when age group $c$ has the same weekday:weekend ratio as the whole sample. In general it does not, because the factors are computed from the sample-wide counts $n_{\text{wd}}$ and $n_{\text{we}}$:

| Age group $c$  | Participants with known day | Raw weekday share | Weighted weekday share $s_c$ | Target $5/7$ |
|----------------|-----------------------------|-------------------|------------------------------|--------------|
| $[0, 5)$       | 89                          | 0.674             | 0.653                        | 0.714        |
| $[5, 15)$      | 196                         | 0.730             | 0.711                        | 0.714        |
| $[15, \infty)$ | 682                         | 0.742             | 0.723                        | 0.714        |
| all            | 967                         | 0.733             | **0.714**                    | 0.714        |

The weights hit $5/7$ exactly across the whole sample, but not within each age group. The under-5s already had too few weekday respondents and get pushed further from the target. A within-group alternative would compute the factors from $n_{c,\text{wd}}$ and $n_{c,\text{we}}$ instead, which gives $s_c = 5/7$ in every age group.

## 3. The general weighting setup

The day-of-week example contains every ingredient of the general setup. This section states it in general, in the order in which socialmixr applies the steps:

1. the weighting factors are multiplied into a **composite weight** $\tilde w_i$;
2. the composite weights are **normalised** (and optionally truncated) within each age group, giving the weights $w_i$;
3. the normalised weights $w_i$ enter the **estimator** $m_{cd}$.

### Weights are products of factors

Every weighting step $k = 1, \dots, K$ assigns each participant a factor $f^{(k)}_i$, and the factors are multiplied together into the composite weight

$$
\tilde w_i = \prod_{k=1}^{K} f^{(k)}_i .
$$

In R, each `weigh*()` call is one step:

```r
survey |>
  assign_age_groups(age_limits = c(0, 5, 15)) |>
  weigh_by_dayofweek() |>          # f^(1): day of week        (Section 4)
  weigh_by_age(uk_pop) |>          # f^(2): age                (Section 4)
  weigh("design_weight") |>        # f^(3): a numeric column   (Section 4)
  compute_matrix(weight_threshold = 3)
```

**Order does not matter.** Multiplication commutes, and the counts used to build each factor depend on the participants, not on the current weights. One exception: `weigh_by_age()` can **drop participants** (Section 5, *`weigh_by_age()` and the open-ended top band*), which changes the counts seen by later steps.

### Normalisation and truncation

Before computing the matrix, `compute_matrix()` rescales the composite weights within each participant age group so that they sum to the number of participants in it. This gives the normalised weights

$$
w_i \;=\; \tilde w_i \cdot \frac{n_c}{\sum_{j \in \mathcal{N}_c} \tilde w_j}, \qquad i \in \mathcal{N}_c,
\qquad\text{so that}\qquad \sum_{i \in \mathcal{N}_c} w_i = n_c .
$$

The socialmixr vignette describes this as making weighted totals equal "known population totals" (citing Kolenikov, 2016). In fact the totals are the **sample** sizes $n_c$, not population sizes.

**Truncation (`weight_threshold`).** Extreme weights can make a few participants dominate an age group. With a threshold $\tau$, after normalisation:

1. **Cap:** $w_i \leftarrow \min(w_i, \tau)$.
2. **Re-normalise once:** $w_i \leftarrow w_i \cdot n_c \big/ \sum_{j \in \mathcal{N}_c} w_j$ for $i \in \mathcal{N}_c$.

The threshold is measured on the normalised scale, where the mean weight in each age group is 1. Re-normalisation happens only once, so weights close to $\tau$ can end up **slightly above** $\tau$.

```r
compute_matrix(survey, weight_threshold = 3)   # default NULL = no truncation
```

### The estimator

The contact matrix is the weighted ratio (Hájek) estimator with the normalised weights $w_i$, computed separately for each participant age group $c$:

$$
m_{cd} \;=\; \frac{\sum_{i \in \mathcal{N}_c} w_i \, y_{id}}{\sum_{i \in \mathcal{N}_c} w_i}.
$$

**Invariance to rescaling within an age group.** For any constants $\gamma_c > 0$, replacing $w_i$ by $\gamma_c w_i$ for all $i \in \mathcal{N}_c$ leaves $m_{cd}$ unchanged, because $\gamma_c$ cancels between numerator and denominator. Normalisation is such a rescaling, with $\gamma_c = n_c / \sum_{j \in \mathcal{N}_c} \tilde w_j$. So without truncation, the estimator gives the same result with the composite weights:

$$
m_{cd} \;=\; \frac{\sum_{i \in \mathcal{N}_c} \tilde w_i \, y_{id}}{\sum_{i \in \mathcal{N}_c} \tilde w_i}.
$$

This is why Section 2 could use the day-of-week factors $f^{\text{dow}}_i$ directly. Normalisation matters only for truncation (above) and for `counts = TRUE` (below).

**Consequences.**
- Weights only change the **mix of participants within** each age group $c$: in Section 2, the mix of weekday and weekend respondents. They never change the relative size of the age groups.
- Population sizes do not enter $m_{cd}$ at all. socialmixr uses them only afterwards, e.g. when the matrix is made reciprocal or converted to per-capita rates.
- Constant factors cancel. Only the ratios between factors within an age group matter: in Section 2 only the ratio $5/2$ matters, not the absolute totals.

**Two caveats on what `compute_matrix()` returns.**
- **`compute_matrix(counts = TRUE)`** skips the division and returns the weighted contact totals $\sum_{i \in \mathcal{N}_c} w_i \, y_{id}$, using the **normalised** weights $w_i$. These are not raw contact counts. The weights in each age group sum to $n_c$, so the totals are on the scale of the sample size, but the weekday/weekend (or other) mix has been reweighted. Unlike $m_{cd}$, this output depends on how the weights are scaled, so the normalisation step really matters here.
- **`compute_matrix(by = c("age", "gender", ...))`** builds a multi-dimensional matrix. Each participant cell (age × gender × …) plays the role of $c$ in the formula above, and likewise for the contact cell and $d$. The normalisation step, however, still groups by **age group only**.

## 4. The weight types

socialmixr provides three weighting functions:

| Function                                               | Purpose                                                                                          |
|--------------------------------------------------------|--------------------------------------------------------------------------------------------------|
| `weigh_by_dayofweek(survey)`                           | weekday/weekend 5 : 2 split (*Day-of-week weights*)                                              |
| `weigh_by_age(survey, pop, ...)`                       | age post-stratification against a reference population (*Age post-stratification*)                 |
| `weigh(survey, by, target = NULL, groups = NULL, ...)` | general mechanism; its behaviour depends on the type of object passed to `target` (*The general `weigh()` function*) |

All three take and return a survey object, so they can be piped. They copy the survey and do not modify the input.

### Day-of-week weights

Covered in full in [Section 2](#2.-Introductory-example-with-day-of-week-weights):

$$
f^{\text{dow}}_i = 5 / n_{\text{wd}} \;\; (i \in \mathcal{W}), \qquad
f^{\text{dow}}_i = 2 / n_{\text{we}} \;\; (i \in \mathcal{E}), \qquad
f^{\text{dow}}_i = 7 / N \;\; (i \in \mathcal{M}).
$$

```r
survey |> weigh_by_dayofweek()
```

### Age post-stratification

**Definition.** For each single-year age $a$ in the observed range of participant ages, $n_a$ is the sample count and $P_a$ the reference population, with $P$ the total population over that range. The factor is the ratio of population share to sample share at the participant's exact age:

$$
f^{\text{age}}_i \;=\; \frac{P_{a_i} / P}{n_{a_i} / N}.
$$

**Interpolating the population to single years.** `pop` is usually given in bands, e.g. 5-year bands with columns `lower.age.limit` and `population`. It is spread **uniformly** over the single years of each band and rounded. For a band $[\ell_k, \ell_{k+1})$ with population $Q_k$:

$$
P_a \;=\; \operatorname{round}\!\left( \frac{Q_k}{\ell_{k+1} - \ell_k} \right), \qquad \ell_k \le a < \ell_{k+1}.
$$

The **last, open-ended band is not split**: its whole population is assigned to its lower limit. This causes the problem described in Section 5 (*`weigh_by_age()` and the open-ended top band*).

**What the age weight actually does.** After normalisation (Section 3, *Normalisation and truncation*), only the variation of $f^{\text{age}}_i$ **within** each age group $c$ matters. Inside $\mathcal{N}_c$ the weight is proportional to $P_{a_i} / n_{a_i}$. So age post-stratification rebalances **single years of age inside each age group**: within $[15, \infty)$, an over-sampled single year gets down-weighted relative to an under-sampled one. Two consequences:

- If the age groups are single years (`age_limits = 0:80`), the age weights cancel completely and have **no effect** on $m_{cd}$.
- The weights do not correct how many participants each age group has relative to the population. Since $m_{cd}$ is already conditional on $c$, this is not needed for the matrix.

**R API.**

```r
uk_pop <- data.frame(
  lower.age.limit = seq(0, 100, by = 5),   # bands should reach well beyond the oldest participant
  population      = ...                    # e.g. from the wpp2024 package
)

survey |>
  assign_age_groups(age_limits = c(0, 5, 15)) |>   # required: creates single-year ages
  weigh_by_age(uk_pop)                              # `...` is passed on to pop_age()
```

**Notes.**
- `weigh_by_age()` aborts unless `assign_age_groups()` has been run and `pop` has columns `lower.age.limit` and `population`.
- Passing a population data frame to `weigh()` (rather than `weigh_by_age()`) still works but has been soft-deprecated since 0.7.0.

### The general `weigh()` function

`weigh_by_dayofweek()` and `weigh_by_age()` are recipes for two specific weights. `weigh()` is the general tool. It does one job: it gives every participant a **weight factor** and multiplies it into the composite weight $\tilde w_i$ (Section 3, *Weights are products of factors*).

```r
weigh(survey, by, target = NULL, groups = NULL, ...)
```

The factor is read off the participant column named by `by`, and **`target` is simply a look-up table** from the values of that column to factors. The table can be written in three ways:

| `target`                         | Look-up table                                                         | Example                                                         |
|----------------------------------|-----------------------------------------------------------------------|-----------------------------------------------------------------|
| two-column data frame            | key column (with the same name as `by`) → value column                | `data.frame(country = c("UK", "DE"), p = c(0.3, 0.7))`          |
| named numeric vector             | `names(target)` → values                                              | `c(child = 0.25, adult = 0.75)`                                 |
| unnamed numeric vector + `groups` | each set of keys in `groups` → the matching element of `target`      | `target = c(5, 2), groups = list(1:5, c(0, 6))`                 |
| `NULL` (default)                 | no table: the `by` column must be numeric and is itself the factor    | `weigh(survey, "design_weight")`                                |

**The one difference between the forms.** With a **data frame**, the value found in the table *is* the factor. With a **vector** (named, or with `groups`), the value is a **total** to be shared equally among all participants who match that key. Each of them gets the value divided by the number of such participants. This is how the day-of-week weights of Section 2 arise: weekday participants share a total of 5, so each gets $5 / n_{\text{wd}}$. `weigh_by_dayofweek()` is exactly the `groups` example in the table.

**R API.**

```r
# NULL: the column is the factor (e.g. design weights supplied with the data)
survey |> weigh("design_weight")

# Data frame: the looked-up value is the factor (e.g. pooling countries by target shares)
country_target <- data.frame(country = c("United Kingdom", "Germany", "Italy"), p = c(0.3, 0.4, 0.3))
polymod |>
  assign_age_groups(age_limits = c(0, 5, 15)) |>
  weigh("country", target = country_target)

# Named vector: children share a total of 0.25, adults a total of 0.75
survey$participants[, agecat := ifelse(part_age < 18, "child", "adult")]
survey |> weigh("agecat", target = c(child = 0.25, adult = 0.75))

# Unnamed vector + groups: weekdays share a total of 5, weekend days a total of 2
survey |> weigh("dayofweek", target = c(5, 2), groups = list(1:5, c(0, 6)))
```

**Notes.**
- **Keys missing from the table.**
  - With a data frame or a named vector, those participants get an `NA` weight, with only a warning. This turns the affected rows of the matrix, often the whole matrix, into `NA` (Section 5, *NA weights propagate*).
  - With `groups`, they get the average weight $\sum_g t_g / N$, where $t_g$ are the elements of `target`. The same applies to missing (`NA`) keys.
  - A group with no participants triggers a warning, and its total is ignored.
- **When the counts are taken.** The number of participants sharing a total is counted over the participants present **when `weigh()` is called**. Filtering the survey before weighting changes the factors; filtering afterwards does not.
- **Factors constant within age groups do nothing.** A factor that is constant within each age group has no effect on $m_{cd}$ (Section 3, *The estimator*). For example, a table keyed on `age.group` does nothing. A look-up only matters when its key varies *within* participant age groups, as `country`, `dayofweek` or `agecat` do.
- **Input checks.** The `NULL` form requires a numeric column. A data frame must have exactly two columns and a numeric value column. `target` and `groups` must have the same length.

## 5. Caveats and limitations

### `weigh_by_age()` and the open-ended top band

As noted in Section 4, *Age post-stratification*, the population of the last band $[\ell_K, \infty)$ is assigned entirely to the single year $\ell_K$, and no population is defined for older single years. When the population is matched to participants:

- participants older than $\ell_K$ are **silently removed** from the survey, and
- participants aged exactly $\ell_K$ get the population of the whole open-ended band, so their weight is hugely inflated.

**Worked example (POLYMOD UK, population bands `c(0, 5, 15, 65)`).** 43 of 1011 participants (ages 66–79) were dropped. Each 65-year-old got a factor of about 13.07, compared with about 0.8 for neighbouring ages.

**Workaround:** supply population data whose bands reach well past the oldest participant, e.g. WPP data up to 100+. The effect then shrinks to centenarians but does not disappear.

### No bootstrap that re-estimates the weights

`compute_matrix()` has no bootstrap. If uncertainty is quantified by resampling participants, the weights should be **rebuilt on each replicate**: the factors depend on sample counts such as $n_{\text{wd}}$, $n_{\text{we}}$ and $n_a$. Reusing weights built once on the full sample ignores the variability that comes from estimating the weights.

### NA weights propagate

The data-frame and named-vector forms of `target` in `weigh()` (Section 4, *The general `weigh()` function*) give NA weights to unmatched keys. In the normalisation step (Section 3, *Normalisation and truncation*), $\sum_{j \in \mathcal{N}_c} \tilde w_j$ becomes NA for every age group containing such a participant, so those rows of $m_{cd}$ become NA. In POLYMOD UK, joining on gender with only `"F"` in the target made **the entire matrix NA**. There is only a warning when `weigh()` is called.

### Not supported

- contact-level weights;
- raking or calibration on several margins at once (e.g. age × sex × region);
- design-based variance (strata, clusters / PSUs);
- weights for repeated measures. Surveys with several observations per participant only trigger a warning.

## 6. References

- Funk S, Willem L, Gruson H, et al. *socialmixr: Social Mixing Matrices for Infectious Disease Modelling.* R package. <https://github.com/epiforecasts/socialmixr>
- socialmixr vignette, section "Participant weights".
- Kolenikov S (2016). Post-stratification or non-response adjustment? *Survey Practice* 9(3): 2809. doi:10.29115/SP-2016-0014.
- Mossong J, Hens N, Jit M, et al. (2008). Social contacts and mixing patterns relevant to the spread of infectious diseases. *PLoS Medicine* 5(3): e74.